# Exploratory Data Analysis

> **Notebook flow:** 01 Setup → **[02 EDA]** → 03 Train → 04 Docker → 05 Kubernetes → 06 Cleanup · 07 Azure Deploy · 08 API Tests

**Purpose**: Understand the raw dataset before modelling. Covers structural inspection, nulls,
class imbalance (target), feature distributions, and correlation analysis.

**Prerequisite:** Complete `01_devcontainer_setup.ipynb` and place your dataset at the path configured in `data.raw_path` in `../config.yaml`.

**Entrypoint**: All paths and parameters are driven by `../config.yaml`.
No values are hardcoded in this notebook.


## 1. Import Required Libraries

In [ ]:
import sys
import os
import warnings

# Allow importing src/ from the notebooks/ directory
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress known harmless warnings; leave data warnings visible.
# Pandas4Warning: select_dtypes("object") deprecation for pandas 3 — safe to ignore on pandas 2.2.x.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

from src.data import load_data
from src.config import load_config

CONFIG_PATH = "../config.yaml"

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_theme(style="whitegrid")

## 2. Load Configuration

In [ ]:
config = load_config(CONFIG_PATH)

TARGET = config["model"]["target_column"]

print("Config loaded:")
print(f"  Raw data path : {config['data']['raw_path']}")
print(f"  Target column : {TARGET}")
print(f"  Test size     : {config['model']['test_size']}")
print(f"  Random state  : {config['model']['random_state']}")

## 3. Load Raw CSV Data

In [ ]:
df = load_data(config, os.path.dirname(CONFIG_PATH))

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head()

## 4. Data Inspection & Validation

In [ ]:
# --- Dtypes overview ---
print("=== Column Types ===")
print(df.dtypes.to_string())

print("\n=== Descriptive Statistics ===")
df.describe(include="all")

In [ ]:
# --- Null analysis ---
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_summary = null_summary[null_summary["null_count"] > 0].sort_values(
    "null_pct", ascending=False
)

if null_summary.empty:
    print("No missing values found.")
else:
    print(f"Columns with missing values (flagged if >5%):\n")
    null_summary["flagged"] = null_summary["null_pct"] > 5
    print(null_summary.to_string())

In [ ]:
# --- Class imbalance ---
class_counts = df[TARGET].value_counts()
class_pct = (class_counts / len(df) * 100).round(2)

print(f"=== Target: '{TARGET}' ===")
for label, count in class_counts.items():
    print(f"  {label}: {count:,}  ({class_pct[label]}%)")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (majority/minority): {imbalance_ratio:.1f}x")
if imbalance_ratio > 3:
    print(
        "  ⚠ Significant imbalance detected — consider SMOTE or class_weight='balanced' during training."
    )

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax, color=["#2196F3", "#FF7043"], edgecolor="black")
ax.set_title(f"Class Distribution — '{TARGET}'")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_xticklabels(class_counts.index, rotation=0)
for i, (count, pct) in enumerate(zip(class_counts, class_pct)):
    ax.text(i, count + 20, f"{pct}%", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
# --- Numeric feature distributions ---
numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")

if len(numeric_cols) == 0:
    print("No numeric features found.")
else:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5), squeeze=False)
    axes = axes.flatten()

    for i, col in enumerate(numeric_cols):
        axes[i].hist(
            df[col].dropna(), bins=40, color="#2196F3", edgecolor="white", alpha=0.8
        )
        axes[i].set_title(col)
        axes[i].set_ylabel("Count")

    # Hide unused axes
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Numeric Feature Distributions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Categorical feature distributions ---
cat_cols = df.select_dtypes(include="object").columns.drop(TARGET, errors="ignore")

if len(cat_cols) == 0:
    print("No categorical features found.")
else:
    n_cols = 2
    n_rows = int(np.ceil(len(cat_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4), squeeze=False)
    axes = axes.flatten()

    for i, col in enumerate(cat_cols):
        counts = df[col].value_counts()
        axes[i].bar(
            counts.index, counts.values, color="#FF7043", edgecolor="white", alpha=0.85
        )
        axes[i].set_title(col)
        axes[i].set_ylabel("Count")
        axes[i].tick_params(axis="x", rotation=30)

    # Hide unused axes
    for j in range(len(cat_cols), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Categorical Feature Distributions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Pearson correlation on numeric columns only (excludes categorical)
CORR_THRESHOLD = 0.8

corr_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[corr_cols].corr()

# Identify strongly correlated pairs (|r| > threshold, upper triangle only)
strong_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > CORR_THRESHOLD:
            strong_pairs.append(
                (corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3))
            )

print(f"Pairs with |r| > {CORR_THRESHOLD}:")
if strong_pairs:
    for col_a, col_b, r in sorted(strong_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {col_a}  ↔  {col_b}  (r = {r})")
else:
    print("  None found — no multicollinearity concerns.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show lower triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Pearson Correlation Matrix (Numeric Features)", fontsize=13)
plt.tight_layout()
plt.show()

from IPython.display import Markdown, display

rows = []

# --- Shape ---
rows.append(("Shape", f"{df.shape[0]:,} rows × {df.shape[1]} columns ({df.shape[1] - 1} features + target)"))

# --- Missing values ---
_missing = null_pct[null_pct > 0].sort_values(ascending=False)
if _missing.empty:
    rows.append(("Missing values", "No missing values found"))
else:
    _parts = [f"`{col}` {pct:.1f}% ({int(null_counts[col])})" for col, pct in _missing.items()]
    _flagged = _missing[_missing >= 5]
    _note = (
        f"⚠️ `{'`, `'.join(_flagged.index)}` flagged for drop (≥5%)"
        if not _flagged.empty
        else "all below 5% threshold, no column flagged for drop"
    )
    rows.append(("Missing values", ", ".join(_parts) + f" — {_note}"))

# --- Class imbalance ---
_majority_label = class_pct.idxmax()
_minority_label = class_pct.idxmin()
_majority_pct   = class_pct.max()
_minority_pct   = class_pct.min()
_ratio          = imbalance_ratio
_imbalance_note = (
    "Significant imbalance. Use `class_weight='balanced'` during training; evaluate with F1/AUC-ROC, not accuracy"
    if _ratio > 3
    else "Mild imbalance. Monitor with F1/AUC-ROC."
)
rows.append((
    "Class imbalance",
    f"`{_majority_label}`: {_majority_pct:.1f}% ({int(class_counts[_majority_label])}), "
    f"`{_minority_label}`: {_minority_pct:.1f}% ({int(class_counts[_minority_label])}) — "
    f"ratio **{_ratio:.1f}:1**. {_imbalance_note}"
))

# --- Suspicious values (age > 100) ---
if "age" in df.columns:
    _suspect = int((df["age"] > 100).sum())
    if _suspect > 0:
        rows.append((
            "Suspicious values",
            f"`age` max = {int(df['age'].max())} ({_suspect} rows > 100) — "
            "likely data entry errors. Cap or remove during preprocessing"
        ))

# --- Skewed distributions: top 3 by absolute skew ---
_skews = df[numeric_cols].skew().abs().sort_values(ascending=False).head(3)
_skew_parts = [f"`{col}` (skew={val:.2f})" for col, val in _skews.items()]
rows.append((
    "Skewed distributions",
    ", ".join(_skew_parts) + " — right-skewed with outliers. Apply log/sqrt transform in `src/features.py`"
))

# --- pdays sentinel encoding ---
if "pdays" in df.columns:
    _sentinel_n = int((df["pdays"] == -1).sum())
    rows.append((
        "`pdays` encoding",
        f"Value `-1` means 'not previously contacted' — {_sentinel_n:,} rows ({_sentinel_n / len(df) * 100:.1f}%). "
        "Acts as sentinel, not a true numeric. Encoded as `contacted_before` binary flag in `src/features.py`"
    ))

# --- Correlated features: strongest pair (regardless of threshold) ---
_corr_abs  = corr_matrix.abs()
_upper_tri = _corr_abs.where(np.triu(np.ones(_corr_abs.shape, dtype=bool), k=1))
_upper_vals = _upper_tri.stack().sort_values(ascending=False)
if not _upper_vals.empty:
    (_ca, _cb) = _upper_vals.index[0]
    _rv   = _upper_vals.iloc[0]
    _sign = corr_matrix.loc[_ca, _cb]
    _strength_note = "⚠️ strong correlation" if _rv > CORR_THRESHOLD else "moderate correlation"
    rows.append((
        "Correlated features",
        f"`{_ca}` ↔ `{_cb}` (r={_sign:.3f}) — {_strength_note}; "
        "no perfect multicollinearity detected; safe to keep both"
    ))
else:
    rows.append(("Correlated features", "No numeric pairs found"))

# --- Render table ---
_header = "## 7. Key Observations\n\n| Area | Observation |\n|---|---|\n"
_table  = "\n".join(f"| {area} | {obs} |" for area, obs in rows)
display(Markdown(_header + _table))


## §8. Config Suggestions (Optional)

Generates a suggested `config.yaml` patch based on EDA outputs — class imbalance, skew, and detected column names.

**Review the printed YAML snippet**, then set `WRITE_CONFIG = True` in the cell below to apply it.

> This only patches the fields it can infer. Values that require domain knowledge (model type, test split, drop_cols) are left unchanged.

In [ ]:
import yaml
from pathlib import Path

# ── Controls ────────────────────────────────────────────────────────────────
WRITE_CONFIG = False   # Set to True to patch config.yaml with the suggestions below
SKEW_THRESHOLD = 1.0   # Columns with |skew| above this are candidates for transforms
# ────────────────────────────────────────────────────────────────────────────

_skew        = df[numeric_cols].skew()
_has_neg     = (df[numeric_cols] < 0).any()
_feature_num = [c for c in numeric_cols if c != TARGET]

# log_transform: positive-skew columns with no negative values
suggested_log_cols = sorted(
    c for c in _feature_num
    if _skew[c] > SKEW_THRESHOLD and not _has_neg[c]
)

# signed_log: columns with negative values + high |skew|  (np.sign(x)*log(|x|+1))
suggested_signed_log_cols = sorted(
    c for c in _feature_num
    if _has_neg[c] and abs(_skew[c]) > SKEW_THRESHOLD
)

# class_weight: recommend "balanced" when majority:minority ratio is significant
suggested_class_weight = "balanced" if imbalance_ratio > 3 else None

# Column-name detection (skip if already in config or not present in data)
_current_features = config.get("features", {})
suggested_age_col   = "age"   if "age"   in df.columns and "age_col"   not in _current_features else None
suggested_pdays_col = "pdays" if "pdays" in df.columns and "pdays_col" not in _current_features else None

# ── Build patch dict ─────────────────────────────────────────────────────────
patch: dict = {}
if suggested_log_cols:
    patch.setdefault("features", {})["log_transform_cols"] = suggested_log_cols
if suggested_signed_log_cols:
    patch.setdefault("features", {})["signed_log_cols"] = suggested_signed_log_cols
if suggested_age_col:
    patch.setdefault("features", {})["age_col"] = suggested_age_col
if suggested_pdays_col:
    patch.setdefault("features", {})["pdays_col"] = suggested_pdays_col
if suggested_class_weight:
    patch.setdefault("model", {})["class_weight"] = suggested_class_weight

# ── Print suggestions ────────────────────────────────────────────────────────
if patch:
    print("=== Suggested config.yaml additions/updates ===\n")
    print(yaml.dump(patch, default_flow_style=False, sort_keys=True))
    print("Review the above, then set WRITE_CONFIG = True to apply.\n")
else:
    print("No suggestions — config.yaml already covers all detected fields.")

# ── Write (opt-in) ───────────────────────────────────────────────────────────
if WRITE_CONFIG and patch:
    _config_path = Path(CONFIG_PATH)
    with open(_config_path) as _f:
        _current = yaml.safe_load(_f)
    for _section, _values in patch.items():
        _current.setdefault(_section, {}).update(_values)
    with open(_config_path, "w") as _f:
        yaml.dump(_current, _f, default_flow_style=False, sort_keys=True)
    print(f"✅ config.yaml updated at {_config_path}")
elif WRITE_CONFIG and not patch:
    print("ℹ️  Nothing to write — config.yaml already up to date.")

---

**Next:** Open **`03_ml_pipeline.ipynb`** to train the model, run the test suite, and verify batch predictions.
